In [ ]:
# @title **Teacher Top-K Truncation Bias Mass Probe (Colab Driver)**
# @markdown Run an analytical probability mass probe across the real prepped dataset to test $K=16, 32, 64, 128, 256, 512$.
# @markdown This measures the exact percentage of Teacher knowledge preserved before setting `top_k_kl`.

# @markdown ### **Probe Configuration:**
DATASET_REPO_ID = "bananamort/the-luau-stack-fim-tokenized" # @param {type:"string"}
DATASET_FILENAME = "fim_train.parquet" # @param {type:"string"}
TEACHER_MODEL = "TorpedoSoftware/Luau-Qwen3-4B-FIM-v0.1" # @param {type:"string"}
NUM_SAMPLES = 200 # @param {type:"integer"}
TEMPERATURE = 2.0 # @param {type:"number"}
HF_TOKEN = "" # @param {type:"string"}

import os, subprocess, sys
from google.colab import userdata

# Retrieve HF Token from Secrets if available
if not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        pass

REPO_URL = "https://github.com/bananamort/luau-qwen2.5-coder-1.5b-distillation.git"
REPO_DIR = "/content/repo"

# 1. Clone repository
if not os.path.exists(REPO_DIR):
    print(f"Cloning repository: {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Updating repository in {REPO_DIR}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)

# 2. Install dependencies
print("Installing dependencies...")
subprocess.run(["pip", "install", "-U", "transformers"], check=True)
subprocess.run(["pip", "install", "datasets", "huggingface_hub", "pyarrow", "numpy"], check=True)

# 3. Execute Mass Probe
cmd = [
    "python", "-u", "src/probe_topk.py",
    "--teacher_model", str(TEACHER_MODEL),
    "--dataset_repo_id", str(DATASET_REPO_ID),
    "--dataset_filename", str(DATASET_FILENAME),
    "--num_samples", str(NUM_SAMPLES),
    "--temperature", str(TEMPERATURE),
]
if HF_TOKEN:
    cmd.extend(["--token", str(HF_TOKEN)])

print(f"Running Teacher Mass Probe over {NUM_SAMPLES} sequences...")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.stdout.close()
if proc.wait() != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd)
